# RetainIQ — Phase 9.4: RAG Answer Generation & MySQL Publishing

## Objective

I connect retrieval to a grounded answer layer and publish the knowledge metadata and query history into MySQL. Generation is optional: when an OpenAI-compatible endpoint is not configured, the notebook returns transparent retrieval-only evidence instead of unsupported claims.

### Notebook presentation rule

For every relevant analytical code cell, I place a **Result & conclusion** markdown cell immediately after it. The analytical code also prints a concise `Result:` line based on the executed data. Setup-only cells are not padded with artificial conclusions.

In [1]:
from pathlib import Path
import json
import os
from datetime import datetime
import pandas as pd
import numpy as np
import requests
import mysql.connector
from IPython.display import display

ROOT = Path('..')
OUTPUT_DIR = ROOT / 'outputs'
CONFIG_DIR = ROOT / 'config'

def print_result(message):
    print(f'Result: {message}')

manifest = json.loads((OUTPUT_DIR / 'retrieval_manifest.json').read_text(encoding='utf-8'))
chunks_df = pd.read_csv(OUTPUT_DIR / 'retrieval_chunks.csv')
vectors = np.load(OUTPUT_DIR / 'retrieval_vectors.npy')
parameters = dict(zip(pd.read_csv(CONFIG_DIR / 'rag_parameters.csv')['parameter'], pd.read_csv(CONFIG_DIR / 'rag_parameters.csv')['value']))

print_result(f'Loaded the RAG index with {len(chunks_df):,} chunks and the {manifest["backend"]} retrieval backend.')

Result: Loaded the RAG index with 8,414 chunks and the tfidf retrieval backend.


In [2]:
def vectorize_query(query):
    if manifest['backend'] == 'sentence_transformers':
        from sentence_transformers import SentenceTransformer
        model = SentenceTransformer(manifest['embedding_model'])
        return np.asarray(model.encode([query], normalize_embeddings=True)[0], dtype=np.float32)
    import joblib
    vectorizer = joblib.load(OUTPUT_DIR / 'tfidf_vectorizer.joblib')
    return vectorizer.transform([query]).toarray()[0].astype(np.float32)

def retrieve(query, top_k=5):
    qv = vectorize_query(query)
    denom = np.linalg.norm(vectors, axis=1) * np.linalg.norm(qv)
    denom = np.where(denom == 0, 1, denom)
    scores = (vectors @ qv) / denom
    order = np.argsort(-scores)[:top_k]
    result = chunks_df.iloc[order].copy()
    result['similarity'] = scores[order]
    result['evidence_id'] = [f'[{i}]' for i in range(1, len(result)+1)]
    return result.reset_index(drop=True)

### Result & conclusion

The generation notebook uses the same retrieval logic as the evaluation notebook, ensuring that the final answer layer is built on the tested index.

In [3]:
llm_enabled = (
    str(
        os.getenv(
            "RETAINIQ_RAG_LLM_ENABLED",
            parameters.get("llm_enabled", "false")
        )
    ).strip().lower()
    == "true"
)

llm_base_url = str(
    os.getenv(
        "RETAINIQ_RAG_LLM_BASE_URL",
        parameters.get(
            "llm_base_url",
            "http://localhost:11434/v1"
        )
    )
).strip().rstrip("/")

llm_model = os.getenv(
    "RETAINIQ_RAG_LLM_MODEL"
)

if not llm_model:
    llm_model = parameters.get("llm_model")

if pd.isna(llm_model):
    llm_model = ""

llm_model = str(llm_model).strip()

print_result(
    f"LLM generation enabled = {llm_enabled}. "
    f"Configured model = '{llm_model or 'none'}'. "
    f"Endpoint = '{llm_base_url}'."
)

Result: LLM generation enabled = True. Configured model = 'qwen3:4b'. Endpoint = 'http://localhost:11434/v1'.


### Result & conclusion

The RAG layer is configured to remain usable without an external API. When an OpenAI-compatible endpoint is explicitly enabled, the same grounded context can be passed to a generative model.

In [4]:
def build_context(retrieved):
    return '\n\n'.join(
        f"{row.evidence_id} Source={row.source_name} | Topic={row.topic}\n{row.chunk_text}"
        for row in retrieved.itertuples(index=False)
    )

def fallback_answer(query, retrieved):
    snippets=[]
    for row in retrieved.itertuples(index=False):
        snippets.append(f"{row.evidence_id} {row.chunk_text}")
    return (
        "I retrieved the following RetainIQ evidence for the question. "
        "A generative model was not used, so I am returning the grounded evidence directly rather than inventing a synthesized answer.\n\n"
        + '\n\n'.join(snippets)
    )

def llm_answer(query, retrieved):
    context = build_context(retrieved)
    system_prompt = (
        "You are the RetainIQ business analytics assistant. "
        "Answer only from the supplied project evidence. "
        "Distinguish observed metrics, calculations, assumptions, and limitations. "
        "Do not invent missing values. Cite evidence identifiers like [1] and [2]."
    )
    user_prompt = f"Question: {query}\n\nProject evidence:\n{context}"
    response = requests.post(
        f"{llm_base_url}/chat/completions",
        headers={'Content-Type': 'application/json'},
        json={
            'model': llm_model,
            'temperature': 0.1,
            'messages': [
                {'role': 'system', 'content': system_prompt},
                {'role': 'user', 'content': user_prompt},
            ]
        },
        timeout=120,
    )
    response.raise_for_status()
    payload = response.json()
    return payload['choices'][0]['message']['content']

def answer_query(query, top_k=5):
    retrieved = retrieve(query, top_k=top_k)
    if llm_enabled and llm_model:
        try:
            answer = llm_answer(query, retrieved)
            mode = 'llm'
        except Exception as exc:
            answer = fallback_answer(query, retrieved)
            mode = f'fallback_after_{type(exc).__name__}'
    else:
        answer = fallback_answer(query, retrieved)
        mode = 'retrieval_only'
    return answer, mode, retrieved

### Result & conclusion

The answer function enforces a grounding boundary: the model receives only retrieved project evidence, and the notebook falls back to transparent evidence output when no compatible LLM is available.

In [5]:
sample_questions = pd.read_csv(CONFIG_DIR / 'rag_sample_questions.csv')
answer_rows=[]
for row in sample_questions.itertuples(index=False):
    answer, mode, retrieved = answer_query(row.question, top_k=int(parameters.get('top_k', 5)))
    answer_rows.append({
        'question_id': row.question_id,
        'question': row.question,
        'mode': mode,
        'answer': answer,
        'evidence_ids': '|'.join(retrieved['evidence_id'].tolist()),
        'top_sources': '|'.join(retrieved['source_name'].head(3).tolist()),
    })

rag_answers = pd.DataFrame(answer_rows)
rag_answers.to_csv(OUTPUT_DIR / 'rag_demo_answers.csv', index=False)
display(rag_answers[['question_id','mode','question','evidence_ids','top_sources']])
print_result(f'Generated {len(rag_answers)} grounded demo answers using the {rag_answers["mode"].iloc[0] if len(rag_answers) else "none"} mode.')

,question_id,mode,question,evidence_ids,top_sources
0,Q01,llm,What is the overall churn rate and how is it d...,[1]|[2]|[3]|[4]|[5],project_methodology|project_methodology|projec...
1,Q02,llm,Which customer segment has the largest histori...,[1]|[2]|[3]|[4]|[5],vw_benchmark_reference|vw_benchmark_reference|...
2,Q03,llm,How does a city's churn rate compare with its ...,[1]|[2]|[3]|[4]|[5],project_methodology|project_methodology|projec...
3,Q04,llm,What does a 20 percent recovery scenario mean ...,[1]|[2]|[3]|[4]|[5],vw_retention_action_framework|project_methodol...
4,Q05,llm,What limitations should I mention when interpr...,[1]|[2]|[3]|[4]|[5],project_methodology|project_methodology|projec...
5,Q06,llm,How does the priority score incorporate ease o...,[1]|[2]|[3]|[4]|[5],project_methodology|vw_retention_strategy_prio...


Result: Generated 6 grounded demo answers using the llm mode.


### Result & conclusion

The demo answers now prove the end-to-end flow: question → retrieval → grounded evidence → answer. When no LLM is configured, the system remains transparent instead of pretending to generate unsupported claims.

In [6]:
MYSQL_CONFIG = {
    'host': 'localhost',
    'port': 3306,
    'user': 'retainiq_user',
    'password': __import__('getpass').getpass('Enter MySQL password for retainiq_user: '),
    'database': 'retainiq',
}

def get_connection():
    return mysql.connector.connect(**MYSQL_CONFIG)

connection = get_connection()
cursor = connection.cursor()

cursor.execute('''
CREATE TABLE IF NOT EXISTS rag_knowledge_documents (
    chunk_id VARCHAR(255) PRIMARY KEY,
    document_id VARCHAR(255) NOT NULL,
    source_name VARCHAR(255) NOT NULL,
    source_type VARCHAR(120) NOT NULL,
    topic VARCHAR(80) NOT NULL,
    chunk_number INT NOT NULL,
    chunk_text MEDIUMTEXT NOT NULL
) ENGINE=InnoDB;
''')

cursor.execute('''
CREATE TABLE IF NOT EXISTS rag_query_log (
    query_id BIGINT AUTO_INCREMENT PRIMARY KEY,
    question_id VARCHAR(40) NULL,
    question_text TEXT NOT NULL,
    answer_mode VARCHAR(80) NOT NULL,
    answer_text MEDIUMTEXT NOT NULL,
    evidence_ids VARCHAR(1000) NULL,
    top_sources VARCHAR(2000) NULL,
    created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
) ENGINE=InnoDB;
''')

connection.commit()
cursor.close()
connection.close()
print_result('Created the RAG knowledge and query-log reporting tables in MySQL.')

Result: Created the RAG knowledge and query-log reporting tables in MySQL.


### Result & conclusion

The structured RAG layer is now ready to persist chunk metadata and answer history in MySQL. Embedding vectors remain file-based because they are a numerical index artifact rather than a reporting table.

In [7]:
connection = get_connection()
cursor = connection.cursor()

cursor.execute('DELETE FROM rag_knowledge_documents')
knowledge_rows = [
    (
        row.chunk_id,
        row.document_id,
        row.source_name,
        row.source_type,
        row.topic,
        int(row.chunk_number),
        row.chunk_text,
    )
    for row in chunks_df.itertuples(index=False)
]
cursor.executemany('''
INSERT INTO rag_knowledge_documents
(chunk_id, document_id, source_name, source_type, topic, chunk_number, chunk_text)
VALUES (%s,%s,%s,%s,%s,%s,%s)
''', knowledge_rows)

cursor.execute('DELETE FROM rag_query_log')
query_rows = [
    (
        row.question_id,
        row.question,
        row.mode,
        row.answer,
        row.evidence_ids,
        row.top_sources,
    )
    for row in rag_answers.itertuples(index=False)
]
cursor.executemany('''
INSERT INTO rag_query_log
(question_id, question_text, answer_mode, answer_text, evidence_ids, top_sources)
VALUES (%s,%s,%s,%s,%s,%s)
''', query_rows)

connection.commit()
cursor.close()
connection.close()
print_result(f'Published {len(knowledge_rows):,} knowledge chunks and {len(query_rows):,} demo queries to MySQL.')

Result: Published 8,414 knowledge chunks and 6 demo queries to MySQL.


### Result & conclusion

The retrieval knowledge and demo interaction layer are now persisted in the RetainIQ database, making the RAG system inspectable through SQL as well as through the notebooks.

In [8]:
connection = get_connection()
cursor = connection.cursor()

cursor.execute('''
CREATE OR REPLACE VIEW vw_rag_knowledge_catalog AS
SELECT chunk_id, document_id, source_name, source_type, topic, chunk_number, chunk_text
FROM rag_knowledge_documents;
''')

cursor.execute('''
CREATE OR REPLACE VIEW vw_rag_query_log AS
SELECT query_id, question_id, question_text, answer_mode, answer_text, evidence_ids, top_sources, created_at
FROM rag_query_log;
''')

cursor.execute('SELECT COUNT(*) FROM rag_knowledge_documents')
knowledge_count = cursor.fetchone()[0]
cursor.execute('SELECT COUNT(*) FROM rag_query_log')
query_count = cursor.fetchone()[0]

connection.commit()
cursor.close()
connection.close()

print_result(f'MySQL validation passed with {knowledge_count:,} knowledge chunks and {query_count:,} logged demo queries.')

Result: MySQL validation passed with 8,414 knowledge chunks and 6 logged demo queries.


### Result & conclusion

Phase 9 now has stable MySQL reporting endpoints for the RAG knowledge catalog and query log. The final system separates the retrieval index, evidence metadata, and interaction history so each layer remains auditable.